# Qdrant Local with External Data Backup

This repository provides a Qdrant Docker image.
The vector database data is stored separately and restored from a backup.

---

## Data Backup (Google Drive)

Download the Qdrant storage backup from Google Drive:

https://drive.google.com/file/d/1mkJbe-Diqye-bUUIWGxRyOp1R_aKXIqK/view?usp=drive_link

After download, extract the archive. You should get a folder structure similar to:

qdrant_storage/
- aliases/
- collections/
- raft_state.json

---

## Run Qdrant

### 1. Create volume
```bash
docker volume create qdrant_data


In [1]:
%pip install -q qdrant-client sentence-transformers tqdm numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\yara2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, re, hashlib
from typing import List, Tuple

import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


C:\Users\yara2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\yara2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# =======================
# 1) CONFIG
# =======================
BASE_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "DATA", "DATA_EXTRACTED")
RFP_DIR = os.path.join(DATA_EXTRACTED_DIR, "RFPs")
TP_DIR  = os.path.join(DATA_EXTRACTED_DIR, "Technical_Proposals")

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", None)

COLLECTION = os.getenv("QDRANT_COLLECTION", "inspira_chunks_v1")

REINDEX = False
CHUNK_CHARS = 700
BATCH = 256

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RFP_DIR:", RFP_DIR)
print("TP_DIR:", TP_DIR)
print("QDRANT_URL:", QDRANT_URL)
print("COLLECTION:", COLLECTION)

if not os.path.isdir(RFP_DIR):
    raise FileNotFoundError(f"RFP_DIR not found: {RFP_DIR}")
if not os.path.isdir(TP_DIR):
    raise FileNotFoundError(f"TP_DIR not found: {TP_DIR}")

# =======================
# 2) HELPERS
# =======================
def read_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def extract_id(filename: str) -> int:
    m = re.search(r"(\d+)", filename)
    if not m:
        raise ValueError(f"Could not extract numeric ID from filename: {filename}")
    return int(m.group(1))

def split_to_chunks(text: str, max_chars: int = 700) -> List[str]:
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks: List[str] = []
    current = ""
    for p in paragraphs:
        if len(current) + len(p) + 1 <= max_chars:
            current += ("\n" + p) if current else p
        else:
            if current:
                chunks.append(current.strip())
            if len(p) <= max_chars:
                current = p
            else:
                for i in range(0, len(p), max_chars):
                    chunks.append(p[i:i + max_chars].strip())
                current = ""
    if current:
        chunks.append(current.strip())
    return chunks

def stable_point_id(doc_type: str, pair_id: int, file_name: str, chunk_id: int) -> int:
    raw = f"{doc_type}:{pair_id}:{file_name}:{chunk_id}".encode("utf-8")
    return int.from_bytes(hashlib.blake2b(raw, digest_size=8).digest(), "big", signed=False)

def load_txt_files(folder: str) -> List[Tuple[int, str, str]]:
    out = []
    for fname in os.listdir(folder):
        if not fname.lower().endswith(".txt"):
            continue
        pid = extract_id(fname)
        text = read_text_file(os.path.join(folder, fname))
        out.append((pid, fname, text))
    return out

# =======================
# 3) EMBEDDING MODEL
# =======================
print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
DIM = model.get_sentence_embedding_dimension()
print("Embedding dim:", DIM)

def embed(texts: List[str]) -> np.ndarray:
    if not texts:
        return np.zeros((0, DIM), dtype="float32")
    vecs = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    return vecs

# =======================
# 4) INIT QDRANT
# =======================
def init_qdrant() -> QdrantClient:
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    existing = [c.name for c in client.get_collections().collections]

    if REINDEX and COLLECTION in existing:
        print("REINDEX=True -> deleting existing collection:", COLLECTION)
        client.delete_collection(COLLECTION)
        existing.remove(COLLECTION)

    if COLLECTION not in existing:
        print("Creating collection:", COLLECTION)
        client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
        )
    else:
        print("Collection already exists:", COLLECTION)

    return client

client = init_qdrant()

# =======================
# 5) BUILD RECORDS (RFP + TP)
# =======================
rfp_files = load_txt_files(RFP_DIR)
tp_files  = load_txt_files(TP_DIR)

print("RFP files found:", len(rfp_files))
print("TP  files found:", len(tp_files))

records: List[Tuple[str, int, str, int, str]] = []  # (doc_type, pair_id, file, chunk_id, text)

for pid, fname, text in rfp_files:
    chunks = split_to_chunks(text, CHUNK_CHARS)
    for cid, ch in enumerate(chunks):
        records.append(("rfp", pid, fname, cid, ch))

for pid, fname, text in tp_files:
    chunks = split_to_chunks(text, CHUNK_CHARS)
    for cid, ch in enumerate(chunks):
        records.append(("tp", pid, fname, cid, ch))

print("Total chunks (RFP+TP):", len(records))
print("Example chunk preview:\n", records[0][4][:300])

# =======================
# 6) UPSERT INTO QDRANT
# =======================
for start in tqdm(range(0, len(records), BATCH), desc="Upserting to Qdrant"):
    batch = records[start:start + BATCH]
    texts = [r[4] for r in batch]
    vecs = embed(texts)

    points = []
    for (doc_type, pid, fname, cid, ch), v in zip(batch, vecs):
        points.append(
            PointStruct(
                id=stable_point_id(doc_type, pid, fname, cid),
                vector=v.tolist(),
                payload={
                    "doc_type": doc_type,   # "rfp" or "tp"
                    "pair_id": int(pid),
                    "file": fname,
                    "chunk_id": int(cid),
                    "text": ch,
                },
            )
        )

    client.upsert(collection_name=COLLECTION, points=points)

print("✅ Done indexing RFP+TP chunks into Qdrant collection:", COLLECTION)

# =======================
# 7) SANITY CHECK
# =======================
info = client.get_collection(COLLECTION)
print("Points count:", info.points_count)
print("Collection status:", info.status)


PROJECT_ROOT: c:\Users\yara2\Desktop\InspiraCore
RFP_DIR: c:\Users\yara2\Desktop\InspiraCore\DATA\DATA_EXTRACTED\RFPs
TP_DIR: c:\Users\yara2\Desktop\InspiraCore\DATA\DATA_EXTRACTED\Technical_Proposals
QDRANT_URL: http://localhost:6333
COLLECTION: inspira_chunks_v1
Loading embedding model...
Embedding dim: 384
Collection already exists: inspira_chunks_v1
RFP files found: 20
TP  files found: 20
Total chunks (RFP+TP): 893
Example chunk preview:
 اسم المنافسه: مشروع الدعم الاستشاري لرفع مستوي النضج في التحول الرقمي وتطوير البنيه الموسسيه
نطاق عمل المشروع
يركز المشروع علي موايمه ممارسات التحول الرقمي و البنيه الموسسيه مع منهجيات وخطط العمل والاستراتيجيات لرفع مستوي نضج التحول الرقمي من خلال الحصول علي خدمات استشاريه لتطوير ممارسات وممكنات الت


Upserting to Qdrant: 100%|██████████| 4/4 [00:06<00:00,  1.73s/it]

✅ Done indexing RFP+TP chunks into Qdrant collection: inspira_chunks_v1
Points count: 893
Collection status: green


In [4]:
from qdrant_client import QdrantClient
client = QdrantClient(url="http://localhost:6333")
print([c.name for c in client.get_collections().collections])
print("inspira_chunks_v1 points:", client.get_collection("inspira_chunks_v1").points_count)


['inspira_chunks_v1', 'tp_chunks_v1']
inspira_chunks_v1 points: 893


# Qdrant Semantic Search – RFP



In [5]:
from qdrant_client.http import models as qm

# Semantic Search Function for RFP / TP Documents
def qdrant_search(
    query: str,
    pair_id: int,
    doc_type: str = "rfp",
    top_k: int = 8,
    min_score: float = 0.40,
    dedup: bool = False
):
    # 1) Convert query text to embedding vector
    qv = embed([query])[0].tolist()

    # 2) Apply metadata filters (project + document type)
    flt = qm.Filter(
        must=[
            qm.FieldCondition(key="pair_id", match=qm.MatchValue(value=int(pair_id))),
            qm.FieldCondition(key="doc_type", match=qm.MatchValue(value=doc_type)),
        ]
    )

    # 3) Vector search using Qdrant (new API)
    res = client.query_points(
        collection_name=COLLECTION,
        query=qv,
        query_filter=flt,
        limit=top_k,
        with_payload=True,
    )

    # 4) Collect results
    out = []
    for p in res.points:
        score = float(p.score or 0.0)
        if score < min_score:
            continue

        payload = p.payload or {}
        out.append({
            "score": score,
            "text": payload.get("text", ""),
            "chunk_id": payload.get("chunk_id"),
            "file": payload.get("file"),
        })

    # 5) Optional: remove duplicate text chunks
    if dedup:
        seen = set()
        deduped = []
        for h in out:
            fingerprint = (h["text"] or "").strip()[:200]
            if not fingerprint or fingerprint in seen:
                continue
            seen.add(fingerprint)
            deduped.append(h)
        out = deduped

    return out

# Test the function
hits = qdrant_search(query="نطاق العمل والمتطلبات الفنية", pair_id=1, doc_type="rfp", top_k=6, min_score=0.0)

for i, h in enumerate(hits, 1):
    print(i, "score=", round(h["score"], 3), "chunk=", h["chunk_id"], "file=", h["file"])
    print(h["text"][:300])
    print("-" * 80) 

1 score= 0.584 chunk= 2 file= Consulting services project for promotional marketing programs for mall products_PR01.txt
مده تنفيذ المشروع 12 اشهر ميالديه تبدا من تاريخ االشعار بالمباشره، يلتزم خاللها المتعاقد باالتي:
تقديم برنامجا زمنيا عام مع العرض الفني يتضمن ترتيب سير العمل والطريقه التي يقترحها لتنفيذ نطاق العمل، وكذلك
علي المتعاقد ان يقدم الي الجهه او ممثل الجهه عندما يطلب منه اي معلومات تفصيليه تتعلق بالترتيبات
--------------------------------------------------------------------------------
2 score= 0.535 chunk= 3 file= Consulting services project for promotional marketing programs for mall products_PR01.txt
تقديم هيكل فريق العمل المنوط به تقديم الخدمات طوال المشروع مع تعيين مدير للمشروع وامكانيه تعيين اصحاب
الخبرات حسب المواضيع المتخصصه.
يلتزم المتعاقد بالمدد السابق ذكرها لتقديم الجدول الزمني وال يمكن تغيير الجدول الزمني للمشروع بعد اعتماده اال في
الحاالت االتيه:
اال توثر مثل هذه التغييرات علي سرعه انه
-----------------------------------------------------------------------------

# Qdrant Semantic Search – Technical Proposals


In [6]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from sentence_transformers import SentenceTransformer

QDRANT_URL = "http://localhost:6333"
COLLECTION = "inspira_chunks_v1"

PAIR_ID_TEST = 4  

TOP_K = 6
MIN_SCORE = 0.40  


client = QdrantClient(url=QDRANT_URL)
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def search_tp(query: str):
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()

    flt = qm.Filter(
        must=[
            qm.FieldCondition(
                key="pair_id",
                match=qm.MatchValue(value=int(PAIR_ID_TEST))
            ),
            qm.FieldCondition(
                key="doc_type",
                match=qm.MatchValue(value="tp")
            ),
        ]
    )

    res = client.query_points(
        collection_name=COLLECTION,
        query=qv,
        query_filter=flt,
        limit=TOP_K,
        with_payload=True,
    )

    hits = []
    for p in res.points:
        score = float(p.score or 0.0)
        if score < MIN_SCORE:
            continue

        payload = p.payload or {}
        hits.append({
            "score": score,
            "file": payload.get("file"),
            "chunk_id": payload.get("chunk_id"),
            "text": payload.get("text",""),
        })

    return hits


if __name__ == "__main__":
    query = "منهجية التنفيذ والجدول الزمني"
    results = search_tp(query)

    for i, h in enumerate(results, 1):
        print(i, "score=", round(h["score"],3), "chunk=", h["chunk_id"], "file=", h["file"])
        print(h["text"][:300])
        print("-" * 80)


1 score= 0.709 chunk= 70 file= Project to provide consulting services for the transfer of electronic climate data_TP_04.txt
المسار الثاني: ادخال بيانات الطقس الساعيه التاريخيه
الفتره الزمنيه للتنفيذ: من الشهر 4 الي الشهر
المسار الثالث: فحص جوده البيانات التاريخيه مناخيا
الفتره الزمنيه للتنفيذ: من الشهر 12 الي الشهر
المسار الرابع: بناء القدرات البشريه
الفتره الزمنيه للتنفيذ: من الشهر 18 الي الشهر
المسار الخامس: اعداد التق
--------------------------------------------------------------------------------
2 score= 0.694 chunk= 31 file= Project to provide consulting services for the transfer of electronic climate data_TP_04.txt
تم تحويل الجدول الزمني بالكامل الي نص قابل للنسخ بدون اي تعديل في المحتوي، فقط نقلته كما هو بشكل نصي واضح:
الجدول الزمني لتنفيذ مهام المشروع
المده الزمنيه لتنفيذ مهام المشروع من تاريخ بدء العمل (ربع سنوي)
المرحله الاولي
تتم خلال: الربع 1 الربع 2 الربع
المرحله الثانيه
تتم خلال: الربع 4 الربع
المرحله 
---------------------------------------------------------------------